In [32]:
import xml.etree.ElementTree as ET

# Load one report
tree = ET.parse("data/ecgen-radiology/1.xml")
root = tree.getroot()

# Print the raw XML to see its structure
import xml.dom.minidom
pretty = xml.dom.minidom.parseString(
    ET.tostring(root)
).toprettyxml(indent="  ")

print(pretty)

<?xml version="1.0" ?>
<eCitation>
  
   
  <meta type="rr"/>
  
   
  <uId id="CXR1"/>
  
   
  <pmcId id="1"/>
  
   
  <docSource>CXR</docSource>
  
   
  <IUXRId id="1"/>
  
   
  <licenseType>open-access</licenseType>
  
   
  <licenseURL>http://creativecommons.org/licenses/by-nc-nd/4.0/</licenseURL>
  
   
  <ccLicense>byncnd</ccLicense>
  
   
  <articleURL/>
  
   
  <articleDate>2013-08-01</articleDate>
  
   
  <articleType>XR</articleType>
  
   
  <publisher>Indiana University</publisher>
  
   
  <title>Indiana University Chest X-ray Collection</title>
  
   
  <note>The data are drawn from multiple hospital systems.</note>
  
   
  <specialty>pulmonary diseases</specialty>
  
   
  <subset>CXR</subset>
  
   
  <MedlineCitation Owner="Indiana University" Status="supplied by publisher">
    
   
      
    <Article PubModel="Electronic">
      
      
         
      <Journal>
        
         
            
        <JournalIssue>
          
            
               
  

In [33]:
import os
import xml.etree.ElementTree as ET

In [34]:
def parse_report(xml_path):
    """
    Given a path to one XML file, extract:
    - the report text (findings + impression)
    - the image IDs that go with it
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()


    findings = ""
    impression = ""
# combine into one text string
    for tag in root.iter("AbstractText"):
        label = tag.get("Label", "")
        text = tag.text or ""

        if label == "FINDINGS":
            findings = text.strip()
        elif label == "IMPRESSION":
            impressionn = text.strip()
    report_text = f"{findings} {impression}".strip()
# Extract image IDs

    image_ids = []
    for img_tag in root.iter("parentImage"):
        img_id = img_tag.get("id")
        if img_id:
            image_ids.append(img_id)

    return report_text, image_ids

text, images = parse_report("data/ecgen-radiology/100.xml")
print("Report text:")
print(text)
print()
print("Image IDs:")
print(images)




    



    

Report text:
Both lungs are clear and expanded. Heart and mediastinum normal.

Image IDs:
['CXR100_IM-0002-1001', 'CXR100_IM-0002-2001']


In [35]:
# Loop through ALL xml files
reports_dir = "data/ecgen-radiology"
images_dir = "data/"

# dataset = [] will hold all (image_path, text) pairs
dataset = []
skipped = 0

for xml_file in os.listdir(reports_dir):
    if not xml_file.endswith(".xml"):
        continue

    xml_path = os.path.join(reports_dir, xml_file)
    report_text, image_ids = parse_report(xml_path)

    # skip if no text was found
    if not report_text:
        skipped +=1 
        continue

    # for each image in this report, create one pair
    for img_id in image_ids:
        img_path = os.path.join(images_dir, f"{img_id}.png")

        # only add if image file actually exists
        if os.path.exists(img_path):
            dataset.append({
                "image_path": img_path,
                "text": report_text
            })
print(f"Total pairs collected: {len(dataset)}")
print(f"Reports skipped (no text): {skipped}")
print()
print("Example pair:")
print(f"  Image: {dataset[0]['image_path']}")
print(f"  Text:  {dataset[0]['text'][:100]}...")

Total pairs collected: 6473
Reports skipped (no text): 530

Example pair:
  Image: data/CXR162_IM-0401-1001.png
  Text:  Heart size normal. Lungs are clear. XXXX are normal. No pneumonia, effusions, edema, pneumothorax, a...


In [36]:
import json

# Save to a JSON file so we dont reparse XML every time
with open("data/dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"saved {len(dataset)} pairs to data/dataset.json")


saved 6473 pairs to data/dataset.json
